# 不使用@tool的方式定义工具

## 1、举例：

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)
# 2、声明一个函数
def get_weather(city: str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("仲恺的天气怎么样？")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 110,
            'prompt_tokens': 266,
            'total_tokens': 376,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 79,
                'rejected_prediction_tokens': None,
                'text_tokens': 110
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 266}
        },
        'model_provider': 'openai',
        'model_name': 'qwen3.7-plus',
        'system_fingerprint': None,
        'id': 'chatcmpl-1a036fbc-acba-9716-828d-2a6238f94710',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f4b12-0566-7bf2-a764-f4644a8fb1f3-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '仲恺'},
            'id': 'call_9103dd4292d643d09a9a6b12',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 266,
        'output_tokens': 110,
        'total_tokens': 376,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 79}
    }
)

## 2、工具描述的各部分详解

### 2.1 了解convert_to_openai_tool

执行 model.bind_tools([get_weather]) ，底层最终会调用 convert_to_openai_tool 生成工具描述。所
以我们可以直接调用后者查看解析后的工具描述。

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city: str):
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

问题：为什么不使用@tool装饰器修饰的函数，也可以理解为工具呢？


查看 convert_to_openai_tool 底层源码：

In [ ]:
# elif isinstance(function, langchain_core.tools.base.BaseTool):
#     oai_function = cast("dict", _format_tool_to_openai_function(function))
# elif callable(function):
#     oai_function = cast(
#         "dict", _convert_python_function_to_openai_function(function)
#     )

相当于加了@tool修饰的函数走上面的分支，没有加@tool修饰的函数走下面的分支，后者会基于函数定
义和docstring生成pydantic模式的描述，然后转换为规范的tool_schema。


## 2.2 description说明

convert_to_openai_tool 会从 docstring(文档字符串) 加载工具的描述信息，上面的案例中，
docstring 为空，所以抽取的 description 为空。

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city: str):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3 参数说明

convert_to_openai_tool 会从 docstring 加载参数说明，这里的 docstring 必须遵循 Google 风格 。


Google 风格 docstring 说明：https://google.github.io/styleguide/pyguide.html


Google 风格 docstring 示例：https://www.sphinx-doc.org/en/master/usage/extensions/example_google.html


Python docstring 通用约定：https://peps.python.org/pep-0257/


基础用法不必完整阅读规范，只需要按照下面的示例仿写即可。

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city: str):
    """
    查询城市的天气

    Args:
        city : 具体的城市

    returns:
        返回城市的天气
    """
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型说明

举例1：正确的

In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：如下的代码运行会报错

要求：如果在docstring中声明了参数的描述，则必须在函数声明处指明参数的类型

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

ValueError: Arg city in docstring not found in function signature.

## 2.5 参数默认值说明

一旦参数设置的默认值，则打印的结果中的`required`字段中就不在包含此函数

举例1：

In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例2：

In [14]:
from langchain_core.utils.function_calling import convert_to_openai_tool


def get_weather(dt : str, city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
        dt : 时间
    """
    return f"{city}天气晴朗~~~"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}